# CP4 · Notebook 02 — Setup

Verifica entorno + highway-env funcional + dataset presente. ~1 min.

In [ ]:
import sys, platform
print(f'Python: {sys.version.split()[0]}  ({platform.system()} {platform.machine()})')
assert sys.version_info >= (3, 10), 'Necesitas Python 3.10+.'

In [ ]:
import numpy as np, torch, gymnasium as gym
import highway_env
print(f'numpy {np.__version__}')
print(f'torch {torch.__version__}  (cuda: {torch.cuda.is_available()})')
print(f'gymnasium {gym.__version__}')
print(f'highway-env {highway_env.__version__}')

## Crear un highway-env de prueba y renderizar 1 frame

Si esto funciona, el entorno está correctamente instalado.

In [ ]:
import matplotlib.pyplot as plt

env = gym.make('highway-v0', config={
    'observation': {
        'type': 'GrayscaleObservation',
        'observation_shape': (84, 84),
        'stack_size': 3,
        'weights': [0.2989, 0.5870, 0.1140],
        'scaling': 1.75,
    },
    'lanes_count': 4,
    'vehicles_count': 20,
})
obs, _ = env.reset(seed=42)
print(f'Observation shape: {obs.shape}  (esperado: (3, 84, 84) — 3 frames stack)')

# Visualizar los 3 frames apilados
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, ax in enumerate(axes):
    ax.imshow(obs[i], cmap='gray')
    ax.set_title(f'Frame {i}'); ax.axis('off')
plt.tight_layout(); plt.show()
env.close()

## Verificar dataset

In [ ]:
from pathlib import Path

DATA_PATH = Path('../datasets/cp4-highway-bc.npz')

if not DATA_PATH.exists():
    print(f'❌ No encuentro {DATA_PATH}')
    print('   Desde la raíz de CP4 ejecuta:  python scripts/generate_dataset.py')
    print('   Tiempo: ~3 min en CPU.')
    raise SystemExit(1)

data = np.load(DATA_PATH)
expected = {
    'train_obs':       (3500, 84, 84, 3),
    'train_actions':   (3500,),
    'val_in_obs':      (750, 84, 84, 3),
    'val_in_actions':  (750,),
    'val_ood_obs':     (750, 84, 84, 3),
    'val_ood_actions': (750,),
}
missing = []
for k, shape in expected.items():
    if k not in data.files:
        missing.append(f'{k} ausente')
        continue
    actual = data[k].shape
    if actual != shape and not (len(actual) == len(shape) and abs(actual[0] - shape[0]) < shape[0] * 0.1):
        missing.append(f'{k}: shape={actual} esperado~{shape}')
    else:
        print(f'  {k:18s}  shape={actual}')

if missing:
    print('\n❌ Dataset con problemas:')
    for m in missing: print(f'   - {m}')
else:
    print('\n✅ Dataset OK')

In [ ]:
if not missing:
    print('✅ Setup OK — listo para 03_dataset.ipynb')
else:
    print('❌ Setup incompleto — revisa warnings arriba')